# Phrase tracker — follow the loudest phrases

Pick the most frequent multi-word phrases and trace their daily trajectory.
Great for spotting a story building, peaking and fading.

*SQL is kept inline in this notebook.*

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Connection settings come from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query and return a DataFrame. Params are passed safely to psycopg."""
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Ready — connected settings for", os.environ["POSTGRES_HOST"])


## The 12 loudest phrases (bigrams and longer)

In [ ]:
top = q("""
    SELECT words, COUNT(*) AS n
    FROM ngrams
    WHERE n_gram >= 2
    GROUP BY words
    ORDER BY n DESC
    LIMIT 50
""")
top

## Their daily trajectories

In [ ]:
phrases = top["words"].tolist()

series = q("""
    SELECT date_trunc('day', timestamp) AS day,
           words,
           COUNT(*)                     AS n
    FROM ngrams
    WHERE words = ANY(%(phrases)s)
    GROUP BY 1, 2
    ORDER BY 1
""", phrases=phrases)

fig = px.line(
    series, x="day", y="n", color="words",
    title="Daily mentions of the top phrases",
)
fig.update_layout(height=520, xaxis_title="", yaxis_title="mentions")
fig.show()

## Small multiples
The same data faceted — one mini-chart per phrase makes individual
rise-and-fall shapes easier to read than an overlapping tangle.

In [ ]:
# ── adjust and re-run ───────────────────────────────────────────────────────
top_n      = 12   # phrases to show (3 … len(phrases))
facet_cols = 3    # subplots per row (1 … 6)
row_h      = 220  # row height in px (100 … 400)
# ────────────────────────────────────────────────────────────────────────────

plot_phrases = phrases[:top_n]
plot_series  = series[series["words"].isin(plot_phrases)]

rows = -(-top_n // facet_cols)
dyn_height        = max(400, rows * row_h)
facet_row_spacing = round(min(0.05, 0.8 / max(rows - 1, 1)), 4)

fig = px.area(
    plot_series, x="day", y="n",
    facet_col="words", facet_col_wrap=facet_cols,
    facet_row_spacing=facet_row_spacing,
    height=dyn_height,
    title=f"Per-phrase trajectories (top {top_n})",
)
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()
